# EDM U-Net: Capacity Sweep (width × depth)

**Purpose.** Prof. Baptista asked directly: *"do the results change when increasing the
number of layers or total number of parameters in the U-Net?"* This notebook sweeps both
knobs at fixed n_train and tracks memorization along training.

- **Width**: `base_channels ∈ {8, 16, 32, 64}` at 3 levels → 64k / 196k / 709k / 2.7M params.
- **Depth**: `num_levels ∈ {2, 4}` at base 16 → 59k / 730k params (3 levels = baseline).
- Fixed n_train = 8, up to 30k steps, log-spaced checkpoints, matched sampler
  (σ_max = 10, 1000 SDE steps), same metrics as the transition notebook.

**Paper prediction** (Baptista et al. Fig. 10–11, 18): more parameters ⇒ closer approximation
of the memorizing empirical-score minimizer; all sizes eventually memorize, larger ones
faster. A genuine capacity-limited generalization regime (small nets never memorizing) would
be the interesting deviation; the per-band metric shows *which scales* memorize first as
capacity grows.

**Runtime:** the 2.7M-param net dominates; expect a few hours for the full grid on MPS.
`SMOKE = True` verifies the pipeline in minutes. Checkpoints saved incrementally per config.

In [ ]:
import sys, os, math, time, copy
import numpy as np
import torch
import matplotlib.pyplot as plt

# -- path setup --
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
src_dir = os.path.join(repo_root, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import diffusion_score_models as score_models
from multiband_data_utils import generate_multiband_dataset_postmask
from memorization_metrics import RingMetricContext
from edm import EDMPrecond, EDMScoreWrapper, train_edm
from unet import SmallUNet, count_parameters
from device_utils import resolve_device

DEVICE = resolve_device()   # cuda > mps > cpu; use resolve_device("cpu") on the personal laptop
print(f'device: {DEVICE}')

In [ ]:
# -- Data generation (identical config to the earlier memorization notebooks) --
components = [
    {"name": "coarse", "length_scale": 2.0,  "s": 2.0, "sigma_sq": 1.0, "band": (0.5, 4.0)},
    {"name": "mid1",   "length_scale": 6.0,  "s": 2.0, "sigma_sq": 1.0, "band": (4.0, 10.0)},
    {"name": "mid2",   "length_scale": 12.0, "s": 2.0, "sigma_sq": 1.0, "band": (10.0, 18.0)},
    {"name": "fine",   "length_scale": 24.0, "s": 2.0, "sigma_sq": 1.0, "band": (18.0, 32.0)},
]
result = generate_multiband_dataset_postmask(
    num_samples=200, grid_size=128, components=components,
    weights=[1.0, 0.8, 0.8, 1.2], seed=42, normalize=True,
)
bands = result.get('bands', {c['name']: c['band'] for c in components})
N = 128
x_all = result['combined']          # kept on CPU; slices moved to DEVICE as needed
ctx = RingMetricContext(N, bands, device=DEVICE)
print(f'x_all: {tuple(x_all.shape)}')

In [ ]:
# -- Evaluation: matched sampler (sigma_max=10, config D of the sigma-fix study) --
VE_SAMPLE = score_models.VE_EDM(sigma_min=0.002, sigma_max=10.0)
N_GEN = 16
N_SDE_STEPS = 1000
N_RAND_REF = 32
LATENT_SEED = 42

@torch.no_grad()
def sample_from(score_fn):
    torch.manual_seed(LATENT_SEED)   # same latents (and step noise stream) for every run
    latents = torch.randn(N_GEN, N*N, device=DEVICE)
    out = VE_SAMPLE.SDEsampler(score_fn, latents, num_steps=N_SDE_STEPS)
    return out.reshape(N_GEN, N, N)

@torch.no_grad()
def pixel_nn_stats(x_gen, x_train):
    '''Relative pixel-space L2 distance to the nearest training field (Baptista-style
    collapse measure). Returns per-sample distances; threshold at plot time.'''
    d = torch.cdist(x_gen.flatten(1), x_train.flatten(1))
    nn_rel = d.min(dim=1).values / x_train.flatten(1).norm(dim=1).mean()
    return nn_rel.cpu()

@torch.no_grad()
def evaluate_checkpoint(precond, x_train):
    wrapper = EDMScoreWrapper(precond, VE_SAMPLE.marginal_prob_std, N, c_tikhonov=0.0).to(DEVICE)
    x_gen = sample_from(wrapper)
    m = ctx.evaluate(x_gen, x_train, n_rand_ref=N_RAND_REF, exclude_nn=True)
    return {
        'coarse_score': m['coarse_score'].mean().item(),
        'fine_score': m['fine_score'].mean().item(),
        'mean_ratio': m['mean_ratio'].cpu(),
        'nn_rel': pixel_nn_stats(x_gen, x_train),
        'samples': x_gen[:2].cpu(),
    }

@torch.no_grad()
def gmm_reference(x_train):
    train_flat = x_train.reshape(x_train.shape[0], -1)
    gmm = score_models.GMM_score(train_flat, VE_SAMPLE.marginal_prob_mean,
                                 VE_SAMPLE.marginal_prob_std)
    x_gen = sample_from(gmm)
    m = ctx.evaluate(x_gen, x_train, n_rand_ref=N_RAND_REF, exclude_nn=True)
    return {
        'coarse_score': m['coarse_score'].mean().item(),
        'fine_score': m['fine_score'].mean().item(),
        'nn_rel': pixel_nn_stats(x_gen, x_train),
    }

results_dir = os.path.join(repo_root, 'results', 'data')
os.makedirs(results_dir, exist_ok=True)
fig_dir = os.path.join(repo_root, 'results', 'figures')
os.makedirs(fig_dir, exist_ok=True)

In [ ]:
# -- Sweep configuration --
SMOKE = False    # True: tiny end-to-end run to verify the pipeline

N_TRAIN = 8
TOTAL_STEPS = 30000
CHECKPOINT_AT = [250, 500, 1000, 2000, 4000, 8000, 16000, 30000]
BATCH_SIZE = 8

CONFIGS = [
    dict(base_channels=8,  emb_dim=64, num_levels=3),
    dict(base_channels=16, emb_dim=64, num_levels=3),   # baseline (= transition notebook)
    dict(base_channels=32, emb_dim=64, num_levels=3),
    dict(base_channels=64, emb_dim=64, num_levels=3),
    dict(base_channels=16, emb_dim=64, num_levels=2),
    dict(base_channels=16, emb_dim=64, num_levels=4),
]

if SMOKE:
    TOTAL_STEPS = 200
    CHECKPOINT_AT = [100, 200]
    N_SDE_STEPS = 50
    N_GEN = 4
    CONFIGS = CONFIGS[:2]

def cfg_name(cfg):
    return f"C{cfg['base_channels']}_L{cfg['num_levels']}"

for cfg in CONFIGS:
    n = count_parameters(SmallUNet(**cfg))
    cfg['n_params'] = n
    print(f"{cfg_name(cfg):>8s}: {n:>10,} params")

x_train = x_all[:N_TRAIN].to(DEVICE)
train_flat = x_train.reshape(N_TRAIN, -1)
ckpt_path = os.path.join(results_dir, 'edm_unet_capacity_checkpoints.pt')

In [ ]:
# -- Training (checkpoints saved incrementally per config) --
all_ckpts = {}
if os.path.exists(ckpt_path):
    all_ckpts = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    print(f'loaded existing checkpoints for {sorted(all_ckpts.keys())}')

for cfg in CONFIGS:
    name = cfg_name(cfg)
    if name in all_ckpts:
        print(f'{name}: already trained, skipping')
        continue
    print(f"===== training {name} ({cfg['n_params']:,} params) =====")
    make_unet = lambda base_channels, emb_dim: SmallUNet(
        base_channels=base_channels, emb_dim=emb_dim, num_levels=cfg['num_levels'])
    t0 = time.time()
    saved = train_edm(train_flat, grid_size=N, total_steps=TOTAL_STEPS,
                      checkpoint_at=CHECKPOINT_AT, base_channels=cfg['base_channels'],
                      emb_dim=cfg['emb_dim'], lr=1e-3, batch_size=BATCH_SIZE,
                      seed=0, device=DEVICE, UNetClass=make_unet)
    all_ckpts[name] = {
        'config': {k: cfg[k] for k in ('base_channels', 'emb_dim', 'num_levels', 'n_params')},
        'ckpts': {step: {'state_dict': {k: v.cpu() for k, v in p.state_dict().items()},
                         'sigma_data': p.sigma_data}
                  for step, p in saved.items()},
    }
    torch.save(all_ckpts, ckpt_path)
    print(f'  {time.time()-t0:.0f}s; saved -> {ckpt_path}')

In [ ]:
# -- Evaluation --
gmm_ref = gmm_reference(x_train)
print(f"GMM ceiling (n_train={N_TRAIN}): coarse={gmm_ref['coarse_score']:.4f} "
      f"fine={gmm_ref['fine_score']:.4f}")

eval_results = {}
for name, entry in all_ckpts.items():
    cfg = entry['config']
    print(f"===== evaluating {name} ({cfg['n_params']:,} params) =====")
    eval_results[name] = {}
    for step, ck in sorted(entry['ckpts'].items()):
        unet = SmallUNet(base_channels=cfg['base_channels'], emb_dim=cfg['emb_dim'],
                         num_levels=cfg['num_levels']).to(DEVICE)
        precond = EDMPrecond(unet, sigma_data=ck['sigma_data']).to(DEVICE)
        precond.load_state_dict(ck['state_dict'])
        precond.eval()
        t0 = time.time()
        r = evaluate_checkpoint(precond, x_train)
        eval_results[name][step] = r
        frac03 = (r['nn_rel'] < 0.3).float().mean().item()
        print(f"  step {step:>6}: coarse={r['coarse_score']:.4f} fine={r['fine_score']:.4f} "
              f"nn_rel(med)={r['nn_rel'].median():.3f} frac<0.3={frac03:.2f} ({time.time()-t0:.0f}s)")

torch.save({'eval_results': eval_results, 'gmm_ref': gmm_ref,
            'configs': {cfg_name(c): c for c in CONFIGS}, 'n_train': N_TRAIN,
            'checkpoint_at': CHECKPOINT_AT,
            'sampler': {'sigma_max': 10.0, 'n_steps': N_SDE_STEPS, 'latent_seed': LATENT_SEED},
            'note': ('EDM SmallUNet capacity sweep at n_train=8: width C in {8,16,32,64} x depth '
                     'L in {2,3,4}; memorization vs training step; matched sampler sigma_max=10.')},
           os.path.join(results_dir, 'edm_unet_capacity_results.pt'))
print('saved -> edm_unet_capacity_results.pt')

In [ ]:
# -- Plots: memorization vs training step per model size --
names = [cfg_name(c) for c in CONFIGS if cfg_name(c) in eval_results]
params = {cfg_name(c): c['n_params'] for c in CONFIGS}
import matplotlib.cm as cm
lo, hi = math.log10(min(params.values())), math.log10(max(params.values()))
def color_of(name):
    return cm.viridis((math.log10(params[name]) - lo) / max(hi - lo, 1e-9))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for ax, key, title in [(axes[0], 'coarse_score', 'coarse band'),
                       (axes[1], 'fine_score', 'fine band')]:
    for name in names:
        steps = sorted(eval_results[name].keys())
        ys = [eval_results[name][s][key] for s in steps]
        ls = '--' if name.endswith(('L2', 'L4')) else '-'
        ax.plot(steps, ys, marker='o', ms=4, ls=ls, color=color_of(name),
                label=f'{name} ({params[name]/1e3:.0f}k)')
    ax.axhline(gmm_ref[key], color='black', lw=0.8, ls=':')
    ax.axhline(1.0, color='gray', lw=0.8, ls='--')
    ax.set_xscale('log')
    ax.set_xlabel('training step')
    ax.set_title(f'{title} (dotted black = GMM ceiling)')
axes[0].set_ylabel('band score (<1 = memorized)')
axes[0].legend(fontsize=7)

ax = axes[2]
for name in names:
    steps = sorted(eval_results[name].keys())
    ys = [(eval_results[name][s]['nn_rel'] < 0.3).float().mean().item() for s in steps]
    ls = '--' if name.endswith(('L2', 'L4')) else '-'
    ax.plot(steps, ys, marker='o', ms=4, ls=ls, color=color_of(name))
ax.set_xscale('log')
ax.set_xlabel('training step')
ax.set_ylabel('fraction rel NN dist < 0.3')
ax.set_title('pixel-space collapse fraction')
fig.suptitle(f'Capacity sweep at n_train={N_TRAIN}: does more width/depth memorize faster?', y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'unet_capacity_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## Notes

- Solid lines: width sweep at 3 levels; dashed: depth variants at C=16. Color = log(params).
- Direct answer for Prof. Baptista: compare curves at fixed training step — the paper
  predicts monotone-in-capacity movement toward the GMM ceiling (larger memorizes faster),
  with everything eventually memorizing. If the small nets plateau above the ceiling at
  30k+ steps, that is architecture-induced regularization worth reporting on its own.
- If a clear size effect appears, a follow-up worth running: the same sweep at n_train=2
  (Baptista's most extreme memorization setting).